In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
import torch

In [2]:
FEATURES = ["ret_1d","ret_2d","ret_3d","ret_5d","ret_10d",
            "price_vs_ma20","ma20_vs_ma50","vol_20d","rsi_14","macd","volume_z"]
SEQ = 60

In [3]:
df = pd.read_parquet("../data/processed/features/TSLA.parquet")

In [5]:
def make_windows(frame,features,seq=60):
    X , y , dates= [] , [], []
    arr= frame[features].values
    tgt= frame["target_1d"].astype(int).values
    for i in range(seq,len(frame)):
        X.append(arr[i-seq:i])
        y.append(tgt[i])
        dates.append(frame.index[i])
    return np.array(X), np.array(y), np.array(dates)        

In [6]:
X, y, dates = make_windows(df, FEATURES)
print("X:", X.shape, "| y:", y.shape)


X: (1140, 60, 11) | y: (1140,)


In [10]:
tr = dates < pd.Timestamp("2024-01-01")
te = dates >= pd.Timestamp("2025-01-01")

mu = X[tr].reshape(-1, 11).mean(0)          # mean/std from TRAIN windows ONLY
sd = X[tr].reshape(-1, 11).std(0) + 1e-8

Xtr = torch.tensor((X[tr]-mu)/sd, dtype=torch.float32)
ytr = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor((X[te]-mu)/sd, dtype=torch.float32)
yte = torch.tensor(y[te], dtype=torch.float32).unsqueeze(1)
print(Xtr.shape, Xte.shape)

torch.Size([507, 60, 11]) torch.Size([381, 60, 11])


In [11]:
from src.dl_models.lstm_model import LSTMClassifier
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch

model = LSTMClassifier()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

train_losses, val_losses = [], []
for epoch in range(30):
    model.train()
    for xb, yb in loader:
        pred = model(xb)              # 1 forward
        loss = loss_fn(pred, yb)      # 2 loss
        opt.zero_grad()               # 3 wipe
        loss.backward()               # 4 gradients
        opt.step()                    # 5 step
    model.eval()
    with torch.no_grad():
        val_losses.append(loss_fn(model(Xte), yte).item())
        train_losses.append(loss.item())
    if epoch % 5 == 0:
        print(f"epoch {epoch}: train {train_losses[-1]:.4f} | val {val_losses[-1]:.4f}")

epoch 0: train 0.6945 | val 0.6932
epoch 5: train 0.6725 | val 0.6960
epoch 10: train 0.7097 | val 0.6980
epoch 15: train 0.7399 | val 0.7212
epoch 20: train 0.6874 | val 0.7526
epoch 25: train 0.6152 | val 0.7654


In [12]:
import plotly.express as px
px.line({"train": train_losses, "val": val_losses}, title="LSTM loss curves").show()

with torch.no_grad():
    acc = ((torch.sigmoid(model(Xte)) > 0.5).int().squeeze().numpy() == y[te]).mean()
print(f"LSTM test acc: {acc:.3f}  | baseline: {max(y[te].mean(), 1-y[te].mean()):.3f}")

LSTM test acc: 0.470  | baseline: 0.512


## Day 11 verdict — LSTM
- LSTM test acc **0.470** vs XGBoost 0.510 vs coin **0.512** → lost to all.
- Loss curves show **classic overfitting**: train-loss fell (0.69→0.62) while
  val-loss rose (0.69→0.77). The model memorized training noise — because there
  is no next-day signal to learn (Day 9 already proved this).
- Confirms across model families (linear, trees, deep sequence): daily direction
  from public price features is ~unlearnable. The rigor — not the score — is the deliverable.
- LSTMs shine where order-signal is real and untraded (language, speech, sensors);
  an efficient market is the opposite of that.

In [13]:
import torch
torch.save(model.state_dict(), "../models/lstm_trend.pt")
print("saved")

saved
